# apertus-eval-prep — paper-matrix **vLLM backend** (Colab ID-5)

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_stability_backend.ipynb)

Runtime → Change runtime type → **T4 GPU**.

This notebook is only the paper-matrix **`backend=vllm`** arm (3 models × 800).

Do **not** Run all. Every session: **cell 1 → 2 → 3 (Drive) → one model sweep**.

**Install note:** Cell 2 matches Colab’s torch CUDA tag (yours is `cu128`) via `uv --torch-backend=…`, then falls back to flexible CUDA-13 runtime wheels if needed. Do not use `vllm==0.10.2` on Python 3.13.

Same Drive folder: `MyDrive/apertus-eval-prep-paper`.


In [ ]:
# Cell 1 — clone or pull
import os
if os.path.exists("pyproject.toml") and os.path.exists("src/apertus_eval_prep"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[viz]"
# vLLM is installed in Cell 2 (CUDA-matched). Do not `pip install vllm` here.
!git log -1 --oneline


In [ ]:
# Cell 2 — pull + GPU + Colab-safe vLLM (rerun every session)
import os, sys, subprocess, glob, re
from pathlib import Path

if Path("pyproject.toml").exists() and Path("src/apertus_eval_prep").exists():
    pass
elif Path("apertus-eval-prep/pyproject.toml").exists():
    os.chdir("apertus-eval-prep")
else:
    raise FileNotFoundError("Run cell 1 first (clone).")

!git pull --ff-only
# Do not install [gpu] here — it pulls a bare mismatched vLLM wheel.
!pip -q install -e ".[viz]"
!git log -1 --oneline

_repo_src = str((Path.cwd() / "src").resolve())
if _repo_src not in sys.path:
    sys.path.insert(0, _repo_src)

import torch
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0), "torch", torch.__version__, "cuda", torch.version.cuda)

def _run(cmd, check=True):
    print("+", " ".join(cmd), flush=True)
    p = subprocess.run(cmd, text=True, capture_output=True)
    if p.stdout.strip():
        print(p.stdout[-2000:], flush=True)
    if p.returncode != 0:
        print(p.stderr[-3000:], flush=True)
        if check:
            raise RuntimeError(f"command failed ({p.returncode}): {' '.join(cmd)}")
    return p

def _pip(*args, check=True):
    return _run([sys.executable, "-m", "pip", *args], check=check)

def _torch_cuda_tag():
    # "12.8" -> "cu128"; "12.4" -> "cu124"
    cuda = torch.version.cuda or ""
    m = re.match(r"(\d+)\.(\d+)", cuda)
    if not m:
        return "cu128"
    return f"cu{m.group(1)}{m.group(2)}"

def _nvidia_lib_dirs():
    dirs = []
    for pattern in (
        "/usr/local/lib/python*/dist-packages/nvidia/**/lib*.so*",
        str(Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages" / "nvidia" / "**" / "lib*.so*"),
        str(Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "dist-packages" / "nvidia" / "**" / "lib*.so*"),
    ):
        for path in glob.glob(pattern, recursive=True):
            dirs.append(str(Path(path).parent))
    return list(dict.fromkeys(dirs))

def _prepend_ld_library_path(dirs):
    if not dirs:
        return
    cur = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join([*dirs, cur] if cur else dirs)
    print("LD_LIBRARY_PATH nvidia dirs:", len(dirs), flush=True)

def _smoke_import():
    for name in list(sys.modules):
        if name == "vllm" or name.startswith("vllm."):
            del sys.modules[name]
    from vllm import LLM, SamplingParams  # noqa: F401
    import vllm
    return vllm

def _install_cuda13_runtime_libs():
    """Best-effort CUDA-13 .so providers for PyPI wheels that link libcudart.so.13."""
    pkgs = [
        "nvidia-cuda-runtime>=13.0,<14",
        "nvidia-cublas>=13.0,<14",
        "nvidia-cuda-nvrtc>=13.0,<14",
        "nvidia-cuda-cupti>=13.0,<14",
    ]
    for pkg in pkgs:
        _pip("install", "-q", "--upgrade", pkg, check=False)
    _prepend_ld_library_path(_nvidia_lib_dirs())

def install_vllm_for_colab():
    """Match Colab torch CUDA (e.g. cu128). Avoid broken exact nvidia pins and ancient vllm==0.10.2."""
    tag = _torch_cuda_tag()
    pin = os.environ.get("APERTUS_VLLM_PIN")  # optional exact version
    print(f"target torch backend tag={tag} pin={pin or 'latest'}", flush=True)

    _pip("uninstall", "-y", "vllm", check=False)

    # 1) Prefer uv's torch-backend selector (picks a CUDA-matched wheel when available).
    _pip("install", "-q", "uv", check=False)
    uv = [sys.executable, "-m", "uv", "pip", "install", "--system"]
    if pin:
        uv.append(f"vllm=={pin}")
    else:
        uv.append("vllm")
    uv.extend([f"--torch-backend={tag}"])
    p = _run(uv, check=False)
    if p.returncode == 0:
        try:
            return _smoke_import()
        except Exception as e:
            print("uv install imported poorly:", e, flush=True)

    # 2) Plain pip + CUDA-13 runtime libs (PyPI default wheel often wants libcudart.so.13).
    print("falling back: CUDA-13 runtime libs + pip install vllm", flush=True)
    _install_cuda13_runtime_libs()
    _pip("uninstall", "-y", "vllm", check=False)
    if pin:
        _pip("install", f"vllm=={pin}")
    else:
        _pip("install", "vllm")
    return _smoke_import()

vllm = install_vllm_for_colab()
print("vllm", getattr(vllm, "__version__", "?"))

if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")


In [ ]:
# Cell 3 — Drive (same folder as HF paper matrix)
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run the clone/pip cell first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
n_partial = len(list(Path("results/runs").glob("*.partial.jsonl")))
print(f"partial checkpoints on disk: {n_partial}", flush=True)

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]
            i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]
            i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    from apertus_eval_prep.sweep import execute_sweep
    print(f"sweep in-process model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    n_skip = sum(1 for p in planned if p["skipped"])
    print({"n_cells": len(planned), "n_skip": n_skip}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --only-factor backend --out-dir results/runs --registry results/registry_paper.jsonl | head -n 30

## ID-5 sweeps — run one model per session

T4 profile: SmolLM2 + Qwen-3B + Phi only (7B vLLM skipped).

In [ ]:
# ID-5 — SmolLM2 backend=vllm (one 800-item run)
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "backend")

In [ ]:
# ID-5 — Qwen-3B backend=vllm (one 800-item run)
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "backend")

In [ ]:
# ID-5 — Phi-3.5 backend=vllm (one 800-item run)
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "backend")

Unpack the Drive zip into the Mac clone. Commit new `results/runs/*.json` and `results/registry_paper.jsonl`. Do not edit numbers.